# Arbeitspaket (AP) 7: Management & Nutzung relationaler Daten

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung, Datenqualität &amp; NULL-Werte</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (hotel.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (hotel.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

Die **fiktive** Schweizer Hotelkette «AlpenStay» betreibt sechs Hotels. Die Datenbank enthält Zimmer, Mitarbeitende, Gäste, Buchungskanäle und die Buchungen der Jahre 2024–2025.

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `hotel.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `hotels`, `rooms`, `employees`, `guests`, `channels`, `bookings`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `hotel_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [ ]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

### Datenbankverbindung zu PostgreSQL

In [ ]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

### Daten nach PostgreSQL laden

In [ ]:
# Tabellen aus der SQLite-Datei hotel.db nach PostgreSQL übertragen
tables = ['hotels', 'rooms', 'employees', 'guests', 'channels', 'bookings']

sqlite_engine = create_engine('sqlite:///hotel.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `hotel.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///hotel.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `hotel.db` werden nicht gewertet.*

### A1 (4 Punkte)

Betrachten Sie das ER-Diagramm von `hotel.db`.

a) Welche **Kardinalität** hat die Beziehung zwischen `hotels` und `rooms`, und welche zwischen `guests` und `rooms`? Begründen Sie mit dem Hotelbetrieb.

b) Wie wird die Beziehung zwischen `guests` und `rooms` im relationalen Modell umgesetzt? Warum gehören die Spalten `nights`, `rate_paid` und `status` genau dorthin — und nicht in `guests` oder `rooms`?

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

In der Tabelle `guests` fallen unter anderem folgende Einträge auf:

```
guestid  firstname  lastname  email                      birthdate   loyalty_level
G0085    Elias      Weber     elias.weber424@example.ch  1993-12-08  Gold
G0613    Elias      Weber     ELIAS.WEBER424@EXAMPLE.CH  1993-12-08  Basic
```

a) Beschreiben Sie das Datenproblem und zwei konkrete Folgen für Auswertungen oder den Betrieb von AlpenStay.

b) Mit welcher Massnahme im **Datenbankschema** hätte man das Problem verhindern können — und warum hätte ein einfacher `UNIQUE`-Constraint auf `email` hier **nicht** genügt?

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

Zwei Analysten zählen die Gäste in `hotel.db`:

```sql
SELECT COUNT(*) FROM guests;                -- Ergebnis: 614
SELECT COUNT(loyalty_level) FROM guests;    -- Ergebnis: 590
```

a) Warum unterscheiden sich die Ergebnisse? Welche Abfrage beantwortet die Frage «Wie viele Gäste sind erfasst?»

b) Eine dritte Abfrage `SELECT COUNT(*) FROM guests WHERE loyalty_level <> 'Gold'` liefert 532. Es gibt 58 Gäste mit `Gold`. Erklären Sie, warum das Ergebnis nicht 614 − 58 = 556 ist.

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `hotel.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Logiserlös (`revenue`) pro Hotel in CHF für die Jahre 2024–2025, gerundet auf 2 Dezimalstellen und absteigend sortiert. Der Logiserlös ist der tatsächlich bezahlte Betrag für alle Übernachtungen aus **bestätigten** Buchungen. Hotels ohne Buchungen müssen nicht erscheinen.

```sql
SELECT h.name,
       ROUND(CAST(SUM(b.rate_paid) AS NUMERIC), 2) AS revenue
FROM hotels h
JOIN rooms r    ON h.hotelid = r.hotelid
JOIN bookings b ON r.roomid  = b.roomid
GROUP BY h.name
ORDER BY revenue DESC;
```

Tatsächliches Ergebnis der SQL-Abfrage:

```
                    name   revenue
0      AlpenStay Zermatt  229608.0
1       AlpenStay Luzern  130543.0
2  AlpenStay Grindelwald  121821.0
3         AlpenStay Bern   93470.0
4        AlpenStay Basel   92526.0
```

Die Buchhaltung meldet für AlpenStay Zermatt einen Logiserlös von rund **CHF 550'000**. Beachten Sie die Spaltenbeschreibungen im Sheet «Description».

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
-- Korrigierte SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_B1

### B2 (8 Punkte)

**Ziel:** Anzahl **bestätigter** Buchungen (`confirmed_bookings`) pro Hotel — für **alle** Hotels. Ein Hotel ohne bestätigte Buchungen soll mit `0` erscheinen.

```sql
SELECT h.name,
       COUNT(b.bookingid) AS confirmed_bookings
FROM hotels h
LEFT JOIN rooms r    ON h.hotelid = r.hotelid
LEFT JOIN bookings b ON r.roomid  = b.roomid
WHERE b.status = 'confirmed'
GROUP BY h.name
ORDER BY confirmed_bookings DESC;
```

Tatsächliches Ergebnis:

```
                    name  confirmed_bookings
0         AlpenStay Bern                 469
1  AlpenStay Grindelwald                 462
2       AlpenStay Luzern                 428
3      AlpenStay Zermatt                 423
4        AlpenStay Basel                 421
```

Die Tabelle `hotels` enthält **6 Hotels**. Das neue Hotel AlpenStay Lugano (Eröffnung 2026) hat bereits Zimmer, aber noch keine Buchungen. Der Analyst ist verwirrt: *«Ich habe doch extra `LEFT JOIN` verwendet!»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 6 Hotels** erscheinen. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
-- Korrigierte SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_B2

---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `hotel.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Die gültigen Werte in `guests.loyalty_level` sind `Basic`, `Silver` und `Gold`. Schreiben Sie eine SQL-Abfrage, die alle Gäste (`guestid`, `firstname`, `lastname`, `loyalty_level`) mit einem ungültigen **oder fehlenden** Wert in `loyalty_level` ausgibt, sortiert nach `guestid`. Begründen Sie Ihre Filterbedingung.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C1

### C2 (4 Punkte)

Wie lange bleiben die Gäste und wie viel bezahlen sie pro Zimmertyp? Geben Sie für **bestätigte** Buchungen pro `room_type` die Anzahl Buchungen (`bookings`), die durchschnittliche Anzahl Nächte (`avg_nights`) und den durchschnittlich bezahlten Preis pro Nacht (`avg_rate`) aus — beide gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_rate`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C2

### C3 (4 Punkte)

Über welchen Buchungskanal wird am häufigsten storniert? Geben Sie pro Kanal `channel_name`, die Anzahl Buchungen (`bookings`), die Anzahl stornierter Buchungen (`cancelled`) und die Stornoquote in Prozent (`cancel_rate_pct`, gerundet auf 1 Dezimalstelle) aus, absteigend sortiert nach `cancel_rate_pct`. Begründen Sie, wie Sie die stornierten Buchungen zählen und worauf Sie bei der Division achten müssen.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C3

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')